# 第 4 章：数据预处理

本 Notebook 是可执行中文版的本地实验版本。它使用与网页相同的确定性样本，演示数据检查、逐月横截面相关性和极端值敏感性。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_PATH = Path("../data/data_ml_web.csv.gz")
data = pd.read_csv(DATA_PATH, parse_dates=["date"])

print(f"记录数：{len(data):,}")
print(f"股票数：{data['stock_id'].nunique():,}")
print(f"日期范围：{data['date'].min():%Y-%m} 至 {data['date'].max():%Y-%m}")
data.head()

## 逐月横截面相关性

总体相关系数会掩盖时间变化。下面在每个月内，分别计算四项特征与未来一个月收益率的相关性。

In [ ]:
feature_cols = [
    "Mkt_Cap_12M_Usd",
    "Pb",
    "Vol1Y_Usd",
    "Mom_11M_Usd",
]

corr_by_month = (
    data.groupby("date")[feature_cols + ["R1M_Usd"]]
    .corr(numeric_only=True)["R1M_Usd"]
    .unstack()[feature_cols]
)

pd.DataFrame({
    "中位数": corr_by_month.median(),
    "下四分位": corr_by_month.quantile(0.25),
    "上四分位": corr_by_month.quantile(0.75),
    "为正的月份占比": (corr_by_month > 0).mean(),
}).round(3).sort_values("中位数")

In [ ]:
ax = corr_by_month.rolling(12, min_periods=6).mean().plot(
    figsize=(10, 5),
    linewidth=1.8,
)
ax.axhline(0, color="#667085", linewidth=1)
ax.set(
    title="12 个月滚动平均相关系数",
    xlabel="日期",
    ylabel="相关系数",
)
ax.legend(frameon=False, ncol=2)
plt.tight_layout()

## 极端值敏感性

修改 `lower_q` 和 `upper_q`，观察不同缩尾阈值对收益分布的影响。

In [ ]:
lower_q = 0.01
upper_q = 0.99

lower, upper = data["R1M_Usd"].quantile([lower_q, upper_q])
clipped = data["R1M_Usd"].clip(lower, upper)

pd.DataFrame({
    "原始收益": data["R1M_Usd"].describe(),
    "缩尾后收益": clipped.describe(),
}).round(4)